# 14 RAGChecker Evaluation

Το notebook προετοιμάζει τα QA και retrieval outputs στη μορφή που απαιτεί το RAGChecker, εκτελεί την αξιολόγηση και αποθηκεύει τα συνοπτικά και αναλυτικά αποτελέσματα.


## 1. Εγκατάσταση

Το κελί εγκαθιστά τις πρόσθετες βιβλιοθήκες όταν δεν υπάρχουν ήδη στο περιβάλλον εκτέλεσης. Σε Colab, Kaggle ή Jupyter ενδέχεται να απαιτηθεί επανεκκίνηση του kernel μετά την εγκατάσταση.


In [ ]:
# %pip install -q ragchecker spacy pandas numpy
# !python -m spacy download en_core_web_sm

print("Uncomment the installation lines above and run them once if needed.")


## 2. Εισαγωγές βιβλιοθηκών


In [ ]:
import os
import json
import ast
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd

from ragchecker import RAGResults, RAGChecker
from ragchecker.metrics import all_metrics


## 3. Ρυθμίσεις

Το notebook χρησιμοποιεί τα αρχεία QA και retrieval που παράγονται στα προηγούμενα στάδια. Το API key διαβάζεται μόνο από μεταβλητή περιβάλλοντος και δεν αποθηκεύεται στον κώδικα.


In [ ]:
# -----------------------------
# ΡΥΘΜΙΣΕΙΣ API KEY / MODEL
# -----------------------------
# Το OPENAI_API_KEY διαβάζεται από το περιβάλλον εκτέλεσης.

EXTRACTOR_NAME = "gpt-4o-mini"
CHECKER_NAME = "gpt-4o-mini"

BATCH_SIZE_EXTRACTOR =4
BATCH_SIZE_CHECKER = 4

# -----------------------------
# ΡΥΘΜΙΣΕΙΣ ΕΙΣΟΔΟΥ
# -----------------------------

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    BASE_DIR = CURRENT_DIR.parent
else:
    BASE_DIR = CURRENT_DIR

DATA_DIR = BASE_DIR / "data"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
QA_DIR = PROCESSED_DIR / "qa_results"
RETRIEVAL_DIR = PROCESSED_DIR / "retrieval_results"
EVAL_DIR = PROCESSED_DIR / "evaluation"

EVAL_DIR.mkdir(parents=True, exist_ok=True)

WORKING_DATASET_CSV_PATH = INTERIM_DIR / "financebench_open_source_working.csv"

DENSE_QA_CSV_PATH = QA_DIR / "rag_qa_results_dense.csv"
HYBRID_QA_CSV_PATH = QA_DIR / "rag_qa_results_hybrid.csv"
RERANK_QA_CSV_PATH = QA_DIR / "rag_qa_results_hybrid_reranked.csv"

DENSE_RETRIEVAL_CSV_PATH = RETRIEVAL_DIR / "retrieval_results_dense.csv"
HYBRID_RETRIEVAL_CSV_PATH = RETRIEVAL_DIR / "retrieval_results_hybrid.csv"
RERANK_RETRIEVAL_CSV_PATH = RETRIEVAL_DIR / "retrieval_results_hybrid_reranked.csv"

RUN_NAME = "dense"   # "dense", "hybrid", "reranked"

QA_PATHS = {
    "dense": DENSE_QA_CSV_PATH,
    "hybrid": HYBRID_QA_CSV_PATH,
    "reranked": RERANK_QA_CSV_PATH,
}

RETRIEVAL_PATHS = {
    "dense": DENSE_RETRIEVAL_CSV_PATH,
    "hybrid": HYBRID_RETRIEVAL_CSV_PATH,
    "reranked": RERANK_RETRIEVAL_CSV_PATH,
}

INPUT_TYPE = "csv"
INPUT_PATH = QA_PATHS[RUN_NAME]
RETRIEVAL_INPUT_PATH = RETRIEVAL_PATHS[RUN_NAME]

TRANSFORMED_JSON_PATH = EVAL_DIR / f"ragchecker_input_{RUN_NAME}.json"
RESULTS_JSON_PATH = EVAL_DIR / f"ragchecker_results_{RUN_NAME}.json"
RESULTS_CSV_PATH = EVAL_DIR / f"ragchecker_metrics_{RUN_NAME}.csv"

# Mapping στηλών εισόδου για τα αρχεία αξιολόγησης
CSV_MAPPING = {
    "question_col": "question",
    "gt_col": "gold_answer",
    "answer_col": "generated_answer",
}

# Στήλες που περιέχουν τα ανακτημένα συμφραζόμενα
RETRIEVED_CONTEXT_COLUMNS = [
    "retrieved_context_1",
    "retrieved_context_2",
    "retrieved_context_3",
    "retrieved_context_4",
    "retrieved_context_5",
]

print("BASE_DIR:", BASE_DIR)
print("INPUT_PATH:", INPUT_PATH)
print("INPUT_PATH exists:", INPUT_PATH.exists())
print("RETRIEVAL_INPUT_PATH:", RETRIEVAL_INPUT_PATH)
print("RETRIEVAL_INPUT_PATH exists:", RETRIEVAL_INPUT_PATH.exists())
print("WORKING_DATASET_CSV_PATH exists:", WORKING_DATASET_CSV_PATH.exists())
print("CSV_MAPPING:", CSV_MAPPING)
print("RETRIEVED_CONTEXT_COLUMNS:", RETRIEVED_CONTEXT_COLUMNS)


## 4. Βοηθητικές συναρτήσεις μετατροπής schema

Το RAGChecker θέλει αυτό το format:

```json
{
  "results": [
    {
      "query_id": "1",
      "query": "...",
      "gt_answer": "...",
      "response": "...",
      "retrieved_context": [
        {"doc_id": "doc_1", "text": "..."},
        {"doc_id": "doc_2", "text": "..."}
      ]
    }
  ]
}
```


In [ ]:
def safe_isna(x: Any) -> bool:
    try:
        return pd.isna(x)
    except Exception:
        return False

def normalize_text(x: Any) -> str:
    if x is None or safe_isna(x):
        return ""
    return str(x).strip()

def parse_retrieved_context(value: Any) -> List[Dict[str, str]]:
    """
    Μετατρέπει διάφορες πιθανές μορφές retrieved context
    σε λίστα από {"doc_id": ..., "text": ...}.
    """
    if value is None or safe_isna(value):
        return []

    # Ήδη λίστα
    if isinstance(value, list):
        out = []
        for i, item in enumerate(value):
            if isinstance(item, dict):
                text = normalize_text(item.get("text", ""))
                doc_id = normalize_text(item.get("doc_id", f"doc_{i+1}"))
                if text:
                    out.append({"doc_id": doc_id or f"doc_{i+1}", "text": text})
            else:
                text = normalize_text(item)
                if text:
                    out.append({"doc_id": f"doc_{i+1}", "text": text})
        return out

    # Αν είναι dict
    if isinstance(value, dict):
        text = normalize_text(value.get("text", ""))
        if text:
            return [{"doc_id": normalize_text(value.get("doc_id", "doc_1")) or "doc_1", "text": text}]
        return []

    # String που μπορεί να είναι JSON / python literal / plain text
    if isinstance(value, str):
        text_value = value.strip()
        if not text_value:
            return []

        # Προσπάθεια JSON
        try:
            parsed = json.loads(text_value)
            return parse_retrieved_context(parsed)
        except Exception:
            pass

        # Προσπάθεια ast.literal_eval
        try:
            parsed = ast.literal_eval(text_value)
            return parse_retrieved_context(parsed)
        except Exception:
            pass

        # Εναλλακτικά: απλό κείμενο chunk
        return [{"doc_id": "doc_1", "text": text_value}]

    # Εναλλακτική περίπτωση
    text = normalize_text(value)
    return [{"doc_id": "doc_1", "text": text}] if text else []


def build_context_from_multiple_columns(row: pd.Series, context_cols: List[str]) -> List[Dict[str, str]]:
    chunks = []
    for i, col in enumerate(context_cols, start=1):
        if col in row.index:
            text = normalize_text(row[col])
            if text:
                chunks.append({"doc_id": f"{col}_{i}", "text": text})
    return chunks


def df_to_ragchecker_records(
    df: pd.DataFrame,
    mapping: Dict[str, str],
    context_cols: Optional[List[str]] = None
) -> List[Dict[str, Any]]:
    context_cols = context_cols or []
    records = []

    required_base = ["query_id", "query", "gt_answer", "response"]
    for field in required_base:
        if mapping.get(field) not in df.columns:
            raise ValueError(f"Missing required column for '{field}': {mapping.get(field)!r}")

    for idx, row in df.iterrows():
        if context_cols:
            retrieved_context = build_context_from_multiple_columns(row, context_cols)
        else:
            rc_col = mapping.get("retrieved_context")
            if rc_col not in df.columns:
                raise ValueError(f"Missing retrieved_context column: {rc_col!r}")
            retrieved_context = parse_retrieved_context(row[rc_col])

        record = {
            "query_id": normalize_text(row[mapping["query_id"]]) or str(idx),
            "query": normalize_text(row[mapping["query"]]),
            "gt_answer": normalize_text(row[mapping["gt_answer"]]),
            "response": normalize_text(row[mapping["response"]]),
            "retrieved_context": retrieved_context,
        }
        records.append(record)
    return records


def load_input_as_ragchecker_json(
    input_path: Path,
    input_type: str,
    mapping: Optional[Dict[str, str]] = None,
    context_cols: Optional[List[str]] = None,
) -> Dict[str, Any]:
    input_type = input_type.lower().strip()

    if input_type == "csv":
        df = pd.read_csv(input_path)
        print(f"Loaded CSV shape: {df.shape}")
        display(df.head(3))
        records = df_to_ragchecker_records(df, mapping=mapping or {}, context_cols=context_cols or [])
        return {"results": records}

    if input_type == "json":
        with open(input_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        # Αν είναι ήδη στο σωστό format
        if isinstance(data, dict) and "results" in data:
            return data

        # Αν είναι λίστα από records
        if isinstance(data, list):
            return {"results": data}

        raise ValueError("Unsupported JSON structure. Expected dict with 'results' or list of records.")

    raise ValueError("INPUT_TYPE must be either 'csv' or 'json'.")


In [ ]:
CSV_MAPPING = {
    "query_id": "financebench_id",
    "query": "question",
    "gt_answer": "expected_answer",
    "response": "generated_answer",
}

RETRIEVED_CONTEXT_COLUMNS = ["context_text"]

print("CSV_MAPPING:", CSV_MAPPING)
print("RETRIEVED_CONTEXT_COLUMNS:", RETRIEVED_CONTEXT_COLUMNS)


## 5. Φόρτωση και μετατροπή δεδομένων


In [ ]:
ragchecker_input = load_input_as_ragchecker_json(
    input_path=INPUT_PATH,
    input_type=INPUT_TYPE,
    mapping=CSV_MAPPING,
    context_cols=RETRIEVED_CONTEXT_COLUMNS,
)

print(f"Number of evaluation examples: {len(ragchecker_input['results'])}")

with open(TRANSFORMED_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(ragchecker_input, f, ensure_ascii=False, indent=2)

print(f"Saved transformed input to: {TRANSFORMED_JSON_PATH}")

ragchecker_input["results"][0] if ragchecker_input["results"] else {}


## 6. Γρήγορο validation του transformed input


In [ ]:
def validate_ragchecker_input(data: Dict[str, Any]) -> pd.DataFrame:
    rows = []
    for item in data.get("results", []):
        rows.append({
            "query_id": item.get("query_id"),
            "query_len": len(item.get("query", "")),
            "gt_answer_len": len(item.get("gt_answer", "")),
            "response_len": len(item.get("response", "")),
            "n_context_chunks": len(item.get("retrieved_context", [])),
            "empty_query": not bool(item.get("query", "").strip()),
            "empty_gt_answer": not bool(item.get("gt_answer", "").strip()),
            "empty_response": not bool(item.get("response", "").strip()),
        })
    return pd.DataFrame(rows)

validation_df = validate_ragchecker_input(ragchecker_input)
display(validation_df.head(10))
display(validation_df.describe(include="all"))


## 7. Εκτέλεση RAGChecker

Το RAGChecker εκτελεί claim extraction και claim checking με LLM. Η εκτέλεση απαιτεί σωστά ορισμένο API key και provider configuration.

Τα διαθέσιμα metric groups είναι:
- `overall_metrics`
- `retriever_metrics`
- `generator_metrics`
- `all_metrics`


In [ ]:

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

print("OPENAI_API_KEY διαθέσιμο:", bool(OPENAI_API_KEY))


In [ ]:
with open(TRANSFORMED_JSON_PATH, "r", encoding="utf-8") as f:
    rag_results = RAGResults.from_json(f.read())

evaluator = RAGChecker(
    extractor_name=EXTRACTOR_NAME,
    checker_name=CHECKER_NAME,
    batch_size_extractor=BATCH_SIZE_EXTRACTOR,
    batch_size_checker=BATCH_SIZE_CHECKER,
)

# Η ενότητα μετρικών μπορεί να οριστεί σε overall_metrics, retriever_metrics ή generator_metrics
metrics_to_run = all_metrics

evaluator.evaluate(rag_results, metrics_to_run)
print("Evaluation completed.")
print(rag_results)


## 8. Αποθήκευση raw αποτελεσμάτων


In [ ]:
rag_results_json_str = rag_results.to_json()

with open(RESULTS_JSON_PATH, "w", encoding="utf-8") as f:
    f.write(rag_results_json_str)

print(f"Saved raw results to: {RESULTS_JSON_PATH}")


## 9. Φόρτωση αποτελεσμάτων και σύνοψη metrics


In [ ]:
with open(RESULTS_JSON_PATH, "r", encoding="utf-8") as f:
    results_data = json.load(f)

results_data.keys()


In [ ]:
SUMMARY_CSV_PATH = EVAL_DIR / f"ragchecker_summary_{RUN_NAME}.csv"

summary_rows = []

metrics_root = results_data.get("metrics", {})

if not isinstance(metrics_root, dict):
    raise ValueError("results_data['metrics'] is not a dict")

for section in ["overall_metrics", "retriever_metrics", "generator_metrics"]:
    section_metrics = metrics_root.get(section, {})
    if isinstance(section_metrics, dict):
        for metric_name, metric_value in section_metrics.items():
            summary_rows.append({
                "metric_group": section,
                "metric_name": metric_name,
                "metric_value": metric_value,
            })

if not summary_rows:
    print("No metrics found under results_data['metrics'].")
else:
    summary_df = pd.DataFrame(summary_rows)
    summary_df = summary_df.sort_values(
        ["metric_group", "metric_name"]
    ).reset_index(drop=True)

    display(summary_df)

    summary_df.to_csv(SUMMARY_CSV_PATH, index=False, encoding="utf-8-sig")
    print(f"Saved summary to: {SUMMARY_CSV_PATH}")


In [ ]:
for k, v in results_data.items():
    print("\nKEY:", k)
    print("TYPE:", type(v))
    if isinstance(v, dict):
        print("SUBKEYS:", list(v.keys())[:20])
    elif isinstance(v, list):
        print("LIST LENGTH:", len(v))
        if len(v) > 0:
            print("FIRST ITEM TYPE:", type(v[0]))


## 10. Αναλυτικά per-example αποτελέσματα

Το ακριβές schema μπορεί να διαφέρει λίγο ανά έκδοση.  
Το παρακάτω cell προσπαθεί να βγάλει όσο πιο χρήσιμο table γίνεται.


In [ ]:
def flatten_example_result(item: Dict[str, Any]) -> Dict[str, Any]:
    row = {
        "query_id": item.get("query_id"),
        "query": item.get("query"),
        "gt_answer": item.get("gt_answer"),
        "response": item.get("response"),
        "n_context_chunks": len(item.get("retrieved_context", []) or []),
    }

    # Αν υπάρχουν per-example metrics
    for possible_key in [
        "overall_metrics",
        "retriever_metrics",
        "generator_metrics",
        "metrics",
        "result",
        "evaluation",
    ]:
        block = item.get(possible_key)
        if isinstance(block, dict):
            for k, v in block.items():
                if isinstance(v, (str, int, float, bool)) or v is None:
                    row[f"{possible_key}.{k}"] = v

    return row

detail_items = results_data.get("results", [])
details_df = pd.DataFrame([flatten_example_result(x) for x in detail_items])

DETAILS_CSV_PATH = EVAL_DIR / f"ragchecker_details_{RUN_NAME}.csv"

results_df = pd.json_normalize(results_data["results"])
display(results_df.head())

results_df.to_csv(DETAILS_CSV_PATH, index=False, encoding="utf-8-sig")
print(f"Saved detailed results to: {DETAILS_CSV_PATH}")

display(details_df.head(10))
details_df.to_csv(DETAILS_CSV_PATH, index=False, encoding="utf-8-sig")
print(f"Saved details to: {DETAILS_CSV_PATH}")


## 11. Ανάλυση αποτυχιών

Αυτό βοηθάει να βρεις δύσκολα queries, μικρό context, ή περιπτώσεις όπου λείπουν retrieved chunks.


In [ ]:
analysis_df = details_df.copy()

if "overall_metrics.f1" in analysis_df.columns:
    display(
        analysis_df.sort_values("overall_metrics.f1", ascending=True)[
            ["query_id", "query", "gt_answer", "response", "n_context_chunks", "overall_metrics.f1"]
        ].head(20)
    )
else:
    display(
        analysis_df.sort_values("n_context_chunks", ascending=True)[
            ["query_id", "query", "gt_answer", "response", "n_context_chunks"]
        ].head(20)
    )


## 12. Προαιρετικό: σύγκριση πολλών runs

Η ενότητα συγκρίνει πολλαπλά `ragchecker_results.json` από dense, hybrid και reranked runs.


In [ ]:
RUN_FILES = {
    # "dense": "outputs_dense/ragchecker_results.json",
    # "hybrid": "outputs_hybrid/ragchecker_results.json",
    # "reranked": "outputs_reranked/ragchecker_results.json",
}

comparison_rows = []

for run_name, path in RUN_FILES.items():
    path = Path(path)
    if not path.exists():
        print(f"Skipping missing file: {path}")
        continue

    with open(path, "r", encoding="utf-8") as f:
        run_data = json.load(f)

    for group in ["overall_metrics", "retriever_metrics", "generator_metrics"]:
        for metric_name, metric_value in run_data.get(group, {}).items():
            comparison_rows.append({
                "run": run_name,
                "metric_group": group,
                "metric_name": metric_name,
                "metric_value": metric_value,
            })

comparison_df = pd.DataFrame(comparison_rows)
if not comparison_df.empty:
    display(comparison_df.sort_values(["metric_group", "metric_name", "run"]))
else:
    print("Add files to RUN_FILES to compare multiple runs.")


## 13. Σημειώσεις χρήσης

Για την αξιολόγηση χρησιμοποιούνται, ανά ερώτηση, τα πεδία `question`, `gold answer`, `generated answer` και τα top-k retrieved chunks. Το notebook μετατρέπει αυτά τα δεδομένα στη μορφή του RAGChecker, εκτελεί την αξιολόγηση και αποθηκεύει summary και detailed outputs για σύγκριση μεταξύ runs.
